n_rows	  n_ids	  min_date	  max_date
896743685	39261	  20230704	  20240703



In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
df = spark.read.table("hive_metastore.default.working_arqlmed")
display(df.limit(5))
df.printSchema()

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("ID").alias("n_ids"),
    F.min("ID_TIME").alias("min_date"),
    F.max("ID_TIME").alias("max_date"),
)
display(summary)

In [0]:
n = df.count()

nulls = df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df.columns
])

nulls_pct = nulls.select(*[
    (F.col(c) / F.lit(n)).alias(c.replace("null_", "pct_null_"))
    for c in nulls.columns
])

display(nulls)
display(nulls_pct)

In [0]:
agg_exprs = [
    F.sum((F.trim(F.col(c)) == F.lit("0")).cast("int")).alias(c)
    for c in df.columns
]

zero_counts = df.agg(*agg_exprs)
display(zero_counts)

In [0]:
(F.col("ID_ARQLMED") / F.lit(n)).alias("ID_ARQLMED")

In [0]:
zero_pct = zero_counts.select(*[
    (F.col(c) / F.lit(n)).alias(c)
    for c in zero_counts.columns
])

display(zero_pct)

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_arqlmed"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
